In [65]:
import json
import time
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import logging


In [66]:

# Global variables
driver = None
wait = None
user_data = {}


In [67]:

def setup_logging():
    """Setup logging configuration"""
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler('job_filler.log'),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)


In [68]:

def create_sample_config(config_file="user_config.json"):
    """Create a sample configuration file"""
    sample_config = {
        "login_credentials": {
            "email": "your.email@example.com",
            "password": "your_password"
        },
        "personal_info": {
            "full_name": "Jane Doe",
            "first_name": "Jane",
            "last_name": "Doe",
            "email": "jane.doe@example.com",
            "phone": "123-456-7890",
            "address": "123 Main St, Cityville, CA 90210",
            "city": "Cityville",
            "state": "California",
            "zip_code": "90210",
            "country": "United States",
            "date_of_birth": "1995-05-15",
            "nationality": "American",
            "linkedin": "https://linkedin.com/in/janedoe",
            "github": "https://github.com/janedoe",
            "portfolio": "https://janedoe.dev"
        },
        "education": {
            "university": "City University",
            "college": "School of Computer Science",
            "degree": "Bachelor of Science",
            "major": "Computer Science",
            "minor": "Mathematics",
            "graduation_year": "2026",
            "expected_graduation": "May 2026",
            "gpa": "3.8",
            "relevant_coursework": "Data Structures, Algorithms, Database Systems, Web Development"
        },
        "experience": {
            "current_position": "Software Engineering Intern",
            "current_company": "Tech Solutions Inc",
            "years_experience": "2",
            "total_experience": "2 years",
            "previous_company": "StartupXYZ",
            "previous_position": "Junior Developer",
            "internship_experience": "Yes",
            "work_history": [
                {
                    "company": "Tech Solutions Inc",
                    "position": "Software Engineering Intern",
                    "start_date": "2023-06",
                    "end_date": "Present",
                    "description": "Developed web applications using React and Node.js"
                }
            ]
        },
        "skills": {
            "programming_languages": "JavaScript, Python, Java, C++",
            "frameworks": "React, Node.js, Django, Spring Boot",
            "databases": "MySQL, PostgreSQL, MongoDB",
            "tools": "Git, Docker, AWS, Jenkins",
            "soft_skills": "Leadership, Communication, Problem-solving, Teamwork",
            "certifications": "AWS Certified Developer, Google Cloud Professional"
        },
        "preferences": {
            "desired_salary": "75000",
            "salary_range_min": "70000",
            "salary_range_max": "85000",
            "work_authorization": "Yes",
            "visa_sponsorship": "No",
            "willing_to_relocate": "Yes",
            "remote_work": "Yes",
            "start_date": "Immediately",
            "availability": "Full-time",
            "notice_period": "2 weeks"
        },
        "files": {
            "resume_path": "C:/Users/YourUsername/Documents/resume.pdf",
            "cover_letter_path": "C:/Users/YourUsername/Documents/cover_letter.pdf",
            "transcript_path": "C:/Users/YourUsername/Documents/transcript.pdf"
        },
        "cover_letter_templates": {
            "software_engineer": "I am writing to express my strong interest in the Software Engineer position...",
            "data_scientist": "I am excited to apply for the Data Scientist role...",
            "default": "I am writing to express my interest in the {job_title} position..."
        }
    }
    
    with open(config_file, 'w') as file:
        json.dump(sample_config, file, indent=2)
    
    print(f"Sample config created at {config_file}. Please update with your information.")


In [69]:

def load_config(config_file="user_config.json"):
    """Load user configuration from JSON file"""
    global user_data
    try:
        with open(config_file, 'r') as file:
            user_data = json.load(file)
            return user_data
    except FileNotFoundError:
        print(f"Config file {config_file} not found. Creating sample config...")
        create_sample_config(config_file)
        return {}


In [70]:

def setup_driver(headless=False):
    """Setup Chrome WebDriver with options"""
    global driver, wait
    
    chrome_options = Options()
    
    if headless:
        chrome_options.add_argument("--headless")
    
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-blink-features=AutomationControlled")
    chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options.add_experimental_option('useAutomationExtension', False)
    
    # Download preferences for file uploads
    prefs = {
        "download.default_directory": os.getcwd(),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True
    }
    chrome_options.add_experimental_option("prefs", prefs)
    
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
        wait = WebDriverWait(driver, 10)
        print("WebDriver initialized successfully")
    except Exception as e:
        print(f"Failed to initialize WebDriver: {e}")
        raise


In [71]:

def detect_and_handle_login():
    """Automatically detect and handle login requirements"""
    global driver, wait, user_data
    
    # Common login indicators
    login_indicators = [
        "input[type='email']",
        "input[name*='email']",
        "input[id*='email']",
        "input[name*='username']",
        "input[id*='username']",
        "input[placeholder*='email']",
        "input[placeholder*='username']",
        "a[href*='login']",
        "a[href*='signin']",
        "button[class*='login']",
        "button[class*='signin']"
    ]
    
    # Check if we're on a login page or need to login
    for indicator in login_indicators:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, indicator)
            if elements and elements[0].is_displayed():
                print("Login required - attempting automatic login...")
                return perform_automatic_login()
        except:
            continue
    
    # Check for login/signin links or buttons
    login_links = [
        "//a[contains(text(), 'Login')]",
        "//a[contains(text(), 'Sign In')]",
        "//a[contains(text(), 'Log In')]",
        "//button[contains(text(), 'Login')]",
        "//button[contains(text(), 'Sign In')]"
    ]
    
    for link_xpath in login_links:
        try:
            login_link = driver.find_element(By.XPATH, link_xpath)
            if login_link.is_displayed():
                print("Found login link - clicking...")
                login_link.click()
                time.sleep(2)
                return perform_automatic_login()
        except:
            continue
    
    print("No login required or login form not detected")
    return True


In [72]:

def perform_automatic_login():
    """Perform automatic login using various strategies"""
    global driver, wait, user_data
    
    if not user_data.get('login_credentials'):
        print("No login credentials found in config")
        return False
    
    email = user_data['login_credentials']['email']
    password = user_data['login_credentials']['password']
    
    # Strategy 1: Standard email/password form
    try:
        # Find email field
        email_selectors = [
            "input[type='email']",
            "input[name*='email']",
            "input[id*='email']",
            "input[name*='username']",
            "input[placeholder*='email']"
        ]
        
        email_field = None
        for selector in email_selectors:
            try:
                email_field = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, selector)))
                break
            except:
                continue
        
        if not email_field:
            print("Email field not found")
            return False
        
        # Enter email
        email_field.clear()
        email_field.send_keys(email)
        email_field.send_keys(Keys.RETURN)
        print("Email entered")
        time.sleep(5)
     
        
        # Find password field  
        password_selectors = [
            "input[type='password']",
            "input[name*='password']",
            "input[id*='password']"
        ]
        
        password_field = None
        for selector in password_selectors:
            try:
                password_field = driver.find_element(By.CSS_SELECTOR, selector)
                if password_field.is_displayed():
                    break
            except:
                continue
        
        if not password_field:
            print("Password field not found")
            return False
        
        # Enter password
        password_field.clear()
        password_field.send_keys(password)
        password_field.send_keys(Keys.RETURN)
        print("Password entered")
        
        
        # Wait for login to complete
        time.sleep(30)
        
        # Check if login was successful
        if check_login_success():
            print("Login successful!")
            return True
        else:
            print("Login may have failed - continuing anyway...")
            return True
            
    except Exception as e:
        print(f"Login failed: {e}")
        return False


In [73]:

def check_login_success():
    """Check if login was successful by looking for common indicators"""
    global driver
    
    # Check for common success indicators
    success_indicators = [
        "//a[contains(text(), 'Logout')]",
        "//a[contains(text(), 'Sign Out')]",
        "//button[contains(text(), 'Logout')]",
        "[class*='profile']",
        "[class*='dashboard']",
        "[class*='account']"
    ]
    
    # Check for failure indicators
    failure_indicators = [
        "//div[contains(text(), 'Invalid')]",
        "//div[contains(text(), 'Error')]",
        "//div[contains(text(), 'incorrect')]",
        "[class*='error']",
        "[class*='alert']"
    ]
    
    # Look for success indicators
    for indicator in success_indicators:
        try:
            if indicator.startswith("//"):
                element = driver.find_element(By.XPATH, indicator)
            else:
                element = driver.find_element(By.CSS_SELECTOR, indicator)
            if element.is_displayed():
                return True
        except:
            continue
    
    # Look for failure indicators
    for indicator in failure_indicators:
        try:
            if indicator.startswith("//"):
                element = driver.find_element(By.XPATH, indicator)
            else:
                element = driver.find_element(By.CSS_SELECTOR, indicator)
            if element.is_displayed():
                return False
        except:
            continue
    
    # If no clear indicators, assume success
    return True


In [74]:

def get_nested_value(data, path):
    """Get nested value from dictionary using dot notation"""
    keys = path.split('.')
    current = data
    
    for key in keys:
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return None
    return current


In [75]:

def smart_fill_field(element, value, field_type="text"):
    """Smart field filling with different strategies"""
    try:
        if field_type == "select":
            select = Select(element)
            # Try to select by visible text first
            options = [option.text.lower() for option in select.options]
            value_lower = str(value).lower()
            
            for option in select.options:
                option_text = option.text.lower()
                if value_lower in option_text or option_text in value_lower:
                    select.select_by_visible_text(option.text)
                    return True
            
            # Try to select by value
            try:
                select.select_by_value(str(value))
                return True
            except:
                pass
                
        elif field_type == "checkbox":
            if str(value).lower() in ['yes', 'true', '1']:
                if not element.is_selected():
                    element.click()
                return True
            elif str(value).lower() in ['no', 'false', '0']:
                if element.is_selected():
                    element.click()
                return True
                    
        else:  # text, email, tel, etc.
            element.clear()
            element.send_keys(str(value))
            return True
            
    except Exception as e:
        print(f"Failed to fill field: {e}")
        return False


In [76]:

def find_and_fill_fields(field_mappings):
    """Find and fill multiple fields based on mappings"""
    global driver, user_data
    filled_count = 0
    
    for data_path, selectors in field_mappings.items():
        value = get_nested_value(user_data, data_path)
        if not value:
            continue
            
        for selector_info in selectors:
            if isinstance(selector_info, str):
                selector = selector_info
                field_type = "text"
            else:
                selector = selector_info["selector"]
                field_type = selector_info.get("type", "text")
            
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    for element in elements:
                        if element.is_displayed() and element.is_enabled():
                            if smart_fill_field(element, value, field_type):
                                filled_count += 1
                                print(f"Filled {selector} with {value}")
                                break
                    break  # Move to next field if we found matching elements
            except Exception as e:
                continue
                
    return filled_count


In [77]:

def fill_basic_info():
    """Fill basic personal information"""
    field_mappings = {
        'personal_info.full_name': [
            'input[name*="name"]:not([name*="first"]):not([name*="last"])',
            'input[id*="fullname"]',
            'input[placeholder*="full name"]'
        ],
        'personal_info.first_name': [
            'input[name*="first"]',
            'input[id*="first"]',
            'input[placeholder*="first name"]'
        ],
        'personal_info.last_name': [
            'input[name*="last"]',
            'input[id*="last"]',
            'input[placeholder*="last name"]'
        ],
        'personal_info.email': [
            'input[type="email"]',
            'input[name*="email"]',
            'input[id*="email"]'
        ],
        'personal_info.phone': [
            'input[type="tel"]',
            'input[name*="phone"]',
            'input[name*="mobile"]',
            'input[id*="phone"]'
        ],
        'personal_info.address': [
            'input[name*="address"]',
            'textarea[name*="address"]',
            'input[id*="address"]'
        ],
        'personal_info.city': [
            'input[name*="city"]',
            'input[id*="city"]'
        ],
        'personal_info.state': [
            {'selector': 'select[name*="state"]', 'type': 'select'},
            'input[name*="state"]'
        ],
        'personal_info.zip_code': [
            'input[name*="zip"]',
            'input[name*="postal"]',
            'input[id*="zip"]'
        ],
        'personal_info.linkedin': [
            'input[name*="linkedin"]',
            'input[placeholder*="linkedin"]'
        ],
        'personal_info.github': [
            'input[name*="github"]',
            'input[placeholder*="github"]'
        ],
        'personal_info.portfolio': [
            'input[name*="portfolio"]',
            'input[name*="website"]',
            'input[placeholder*="portfolio"]'
        ]
    }
    
    return find_and_fill_fields(field_mappings)


In [78]:

def fill_education():
    """Fill education information"""
    field_mappings = {
        'education.university': [
            'input[name*="university"]',
            'input[name*="school"]',
            {'selector': 'select[name*="university"]', 'type': 'select'},
            'input[id*="university"]'
        ],
        'education.degree': [
            {'selector': 'select[name*="degree"]', 'type': 'select'},
            'input[name*="degree"]'
        ],
        'education.major': [
            'input[name*="major"]',
            {'selector': 'select[name*="major"]', 'type': 'select'},
            'input[name*="field"]'
        ],
        'education.gpa': [
            'input[name*="gpa"]',
            'input[id*="gpa"]'
        ],
        'education.graduation_year': [
            {'selector': 'select[name*="graduation"]', 'type': 'select'},
            'input[name*="graduation"]',
            {'selector': 'select[name*="year"]', 'type': 'select'}
        ]
    }
    
    return find_and_fill_fields(field_mappings)


In [79]:

def fill_experience():
    """Fill work experience information"""
    field_mappings = {
        'experience.current_company': [
            'input[name*="company"]',
            'input[name*="employer"]',
            'input[id*="company"]'
        ],
        'experience.current_position': [
            'input[name*="position"]',
            'input[name*="title"]',
            'input[name*="job"]'
        ],
        'experience.years_experience': [
            {'selector': 'select[name*="experience"]', 'type': 'select'},
            'input[name*="experience"]',
            {'selector': 'select[name*="years"]', 'type': 'select'}
        ]
    }
    
    return find_and_fill_fields(field_mappings)


In [80]:

def fill_preferences():
    """Fill job preferences and authorization"""
    field_mappings = {
        'preferences.work_authorization': [
            {'selector': 'select[name*="authorization"]', 'type': 'select'},
            {'selector': 'input[name*="authorization"][type="checkbox"]', 'type': 'checkbox'}
        ],
        'preferences.visa_sponsorship': [
            {'selector': 'select[name*="visa"]', 'type': 'select'},
            {'selector': 'select[name*="sponsor"]', 'type': 'select'},
            {'selector': 'input[name*="visa"][type="checkbox"]', 'type': 'checkbox'}
        ],
        'preferences.willing_to_relocate': [
            {'selector': 'select[name*="relocate"]', 'type': 'select'},
            {'selector': 'input[name*="relocate"][type="checkbox"]', 'type': 'checkbox'}
        ],
        'preferences.remote_work': [
            {'selector': 'select[name*="remote"]', 'type': 'select'},
            {'selector': 'input[name*="remote"][type="checkbox"]', 'type': 'checkbox'}
            
        ],
        'preferences.desired_salary': [
            'input[name*="salary"]',
            'input[name*="compensation"]'
        ]
    }
    
    return find_and_fill_fields(field_mappings)


In [81]:

def upload_resume():
    """Upload resume file"""
    global driver, user_data
    
    if not user_data.get('files', {}).get('resume_path'):
        print("No resume path specified")
        return False
        
    resume_path = user_data['files']['resume_path']
    if not os.path.exists(resume_path):
        print(f"Resume file not found: {resume_path}")
        return False
        
    file_selectors = [
        'input[type="file"][name*="resume"]',
        'input[type="file"][name*="cv"]',
        'input[type="file"][id*="resume"]',
        'input[type="file"]'
    ]
    
    for selector in file_selectors:
        try:
            file_inputs = driver.find_elements(By.CSS_SELECTOR, selector)
            for file_input in file_inputs:
                # Check if this is likely a resume upload field
                context = file_input.get_attribute('name') + ' ' + file_input.get_attribute('id')
                if 'resume' in context.lower() or 'cv' in context.lower() or len(file_inputs) == 1:
                    file_input.send_keys(resume_path)
                    print(f"Resume uploaded: {resume_path}")
                    return True
        except Exception as e:
            continue
            
    print("No resume upload field found")
    return False


In [82]:

def generate_cover_letter(job_title="the position"):
    """Generate and fill cover letter"""
    global driver, user_data
    
    name = user_data.get('personal_info', {}).get('full_name', 'Your Name')
    experience = user_data.get('experience', {}).get('years_experience', 'relevant')
    skills = user_data.get('skills', {}).get('programming_languages', 'technical skills')
    university = user_data.get('education', {}).get('university', 'university')
    major = user_data.get('education', {}).get('major', 'my field')
    
    # Check for job-specific template
    templates = user_data.get('cover_letter_templates', {})
    job_key = job_title.lower().replace(' ', '_')
    
    if job_key in templates:
        cover_letter = templates[job_key].format(
            job_title=job_title,
            name=name,
            experience=experience,
            skills=skills,
            university=university,
            major=major
        )
    else:
        cover_letter = f"""Dear Hiring Manager,

I am writing to express my strong interest in the {job_title} position at your company. With {experience} years of experience and expertise in {skills}, I am confident that I would be a valuable addition to your team.

My background in {major} from {university} has provided me with a solid foundation in the technical skills required for this role. In my current position, I have developed strong problem-solving abilities and gained hands-on experience with various technologies and frameworks.

I am particularly excited about this opportunity because it aligns perfectly with my career goals and passion for technology. I am eager to contribute to your team's success and would welcome the opportunity to discuss how my skills and enthusiasm can benefit your organization.

Thank you for considering my application. I look forward to hearing from you soon.

Sincerely,
{name}"""

    # Try to find and fill cover letter fields
    cover_letter_selectors = [
        'textarea[name*="cover"]',
        'textarea[id*="cover"]',
        'textarea[placeholder*="cover"]',
        'textarea[name*="letter"]',
        'textarea[placeholder*="motivation"]'
    ]
    
    for selector in cover_letter_selectors:
        try:
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if elements:
                elements[0].clear()
                elements[0].send_keys(cover_letter)
                print("Cover letter generated and filled")
                return True
        except Exception as e:
            continue
            
    print("No cover letter field found")
    return False


In [83]:

def fill_application(application_url, job_title=""):
    """Main function to fill job application"""
    global driver
    
    logger = setup_logging()
    
    try:
        # Load configuration
        load_config()
        
        # Setup browser
        setup_driver()
        options = Options()
        options.add_argument(r"user-data-dir=C:\Users\Iqra Iqbal\Desktop\Bisma")

        # Navigate to application
        print(f"Navigating to application: {application_url}")
        driver.get(application_url)
        time.sleep(3)
        
        # Detect and handle login automatically
        detect_and_handle_login()
        
        # Wait a bit after login
        time.sleep(3)
        
        # Fill different sections
        print("Filling basic information...")
        basic_filled = fill_basic_info()
        
        print("Filling education information...")
        education_filled = fill_education()
        
        print("Filling experience information...")
        experience_filled = fill_experience()
        
        print("Filling preferences...")
        preferences_filled = fill_preferences()
        
        # Upload resume
        print("Uploading resume...")
        resume_uploaded = upload_resume()
        
        # Generate cover letter
        print("Generating cover letter...")
        cover_letter_filled = generate_cover_letter(job_title)
        
        total_filled = basic_filled + education_filled + experience_filled + preferences_filled
        
        print(f"\n=== APPLICATION FILLING COMPLETED ===")
        print(f"Fields filled: {total_filled}")
        print(f"Resume uploaded: {resume_uploaded}")
        print(f"Cover letter filled: {cover_letter_filled}")
        print("=====================================")
        
        # Keep browser open for manual review
        input("Press Enter to close the browser...")
        
    except Exception as e:
        print(f"Error filling application: {e}")
        raise
    finally:
        if driver:
            driver.quit()


In [84]:

def fill_multiple_applications(applications):
    """Fill multiple applications in sequence"""
    for i, app in enumerate(applications, 1):
        try:
            print(f"\n=== Processing Application {i}/{len(applications)} ===")
            fill_application(
                application_url=app['url'],
                job_title=app.get('job_title', '')
            )
            print(f"Application {i} completed")
            
            if i < len(applications):
                wait_time = 10
                print(f"Waiting {wait_time} seconds before next application...")
                time.sleep(wait_time)
                
        except Exception as e:
            print(f"Failed to fill application {app['url']}: {e}")
            continue


In [85]:

# Example usage
if __name__ == "__main__":
    # Single application example
    fill_application(
        application_url="https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform",
        job_title="Software Engineer"
    )
    
    # Multiple applications example
    # applications = [
    #     {
    #         "url": "https://company1.com/apply",
    #         "job_title": "Software Engineer"
    #     },
    #     {
    #         "url": "https://company2.com/careers/apply", 
    #         "job_title": "Data Scientist"
    #     }
    # ]
    # fill_multiple_applications(applications)
    

WebDriver initialized successfully
Navigating to application: https://docs.google.com/forms/d/e/1FAIpQLSdiDl_sgvqMFtOv45fUCVegya6IbqattwhpgFAe9im3PmD_Xw/viewform
No login required or login form not detected
Filling basic information...
Filling education information...
Filling experience information...
Filling preferences...
Uploading resume...
Resume file not found: C:UsersIqra IqbalDesktopTaha Ahmed.pdf
Generating cover letter...
No cover letter field found

=== APPLICATION FILLING COMPLETED ===
Fields filled: 0
Resume uploaded: False
Cover letter filled: False
